# Clusterização de Regimes Climáticos de Recife (PE) com K-Means

Este notebook consolida a pipeline reproduzível de agrupamento não-supervisionado (**K-Means**) para os registros horários da estação automática do INMET em **Recife (A301)** com dados de 2018 a 2021 (`../dados/df_recife_datetime.csv`), integrando os achados dos notebooks exploratórios anteriores (`Limpeza_DataFrame.ipynb`, `analise_variaveis.ipynb`, `analise_chuva.ipynb`, `AED tempo para variação meteorológica.ipynb` e `clusterizacao_gmm.ipynb`).

O notebook está estruturado na seguinte sequência:
1. **Seleção das variáveis** — definição explícita do conjunto de features e exclusão de chuva e identificadores.
2. **Justificativa das escolhas** — avaliação quantitativa e física de relevância, redundância e dados ausentes.
3. **Tratamento e encoding** — verificação de tipos, tratamento de ausentes e codificação cíclica de hora e mês.
4. **Comparação de normalização** — comparação entre dados sem escala, `StandardScaler` e `MinMaxScaler`.
5. **Decisão sobre PCA** — avaliação comparativa com e sem PCA e registro da decisão técnica.
6. **Fechamento da pipeline inicial** — organização sequencial e reproduzível (seleção, tratamento, normalização, escolha de $k$, treino e interpretação).
7. **Validação e registro de evidências** — checagens finais de integridade e execução completa sem erros.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_samples,
    silhouette_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler

RANDOM_STATE = 42
CAMINHO_CSV = "../dados/df_recife_datetime.csv"

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)


---
## 1. Seleção das Variáveis

Antes de treinar um algoritmo baseado em distância euclidiana, definimos as colunas de entrada e as colunas que ficam de fora.

### Revisão do dataset
- A base consolidada `df_recife_datetime.csv` contém a coluna `datetime`, as medições meteorológicas e `ESTACAO` como metadado.
- Chuva fica fora do K-Means e serve somente para validar os grupos depois do ajuste.
- Timestamps e metadados não entram diretamente na distância; hora local e mês são representados por codificações cíclicas.

### Grupos de colunas
- **Identificadores e metadados excluídos, quando presentes**: `Unnamed: 0`, `Data`, `Hora UTC`, `datetime`, `ESTACAO`.
- **Alvo excluído do clustering**: `PRECIPITAÇÃO TOTAL, HORÁRIO (mm)`.
- **Variáveis excluídas por dados ausentes ou redundância**: vento e extremos máximos/mínimos.
- **Features do K-Means**: quatro medições físicas e `hora_sin`, `hora_cos`, `mes_sin`, `mes_cos`.


In [ ]:
# Leitura bruta como texto para preservar exatamente a formatação original antes do tratamento
df_bruto = pd.read_csv(CAMINHO_CSV, dtype=str)
print(f"Dimensões originais do dataset: {df_bruto.shape[0]} linhas x {df_bruto.shape[1]} colunas")

print("\nColunas originais disponíveis:")
for idx, col in enumerate(df_bruto.columns):
    print(f"  [{idx:02d}] {col}")

# Registro explícito dos grupos de variáveis
IDENTIFICADORES_EXCLUIDOS = ["Unnamed: 0", "Data", "Hora UTC", "datetime", "ESTACAO"]
VARIAVEL_ALVO_EXCLUIDA = "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)"

VARIAVEIS_VENTO_EXCLUIDAS = [
    "VENTO, DIREÇÃO HORARIA (gr) (° (gr))",
    "VENTO, RAJADA MAXIMA (m/s)",
    "VENTO, VELOCIDADE HORARIA (m/s)",
]

VARIAVEIS_EXTREMOS_EXCLUIDAS = [
    "PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)",
    "PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)",
    "TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)",
    "UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)",
    "UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)",
]

VARIAVEIS_FISICAS_SELECIONADAS = [
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",
    "TEMPERATURA DO PONTO DE ORVALHO (°C)",
    "UMIDADE RELATIVA DO AR, HORARIA (%)",
]

VARIAVEIS_TEMPORAIS_CICLICAS = ["hora_sin", "hora_cos", "mes_sin", "mes_cos"]

FEATURES_CLUSTER = VARIAVEIS_FISICAS_SELECIONADAS + VARIAVEIS_TEMPORAIS_CICLICAS

# Garantias formais de exclusão
assert VARIAVEL_ALVO_EXCLUIDA not in FEATURES_CLUSTER, "A chuva não pode entrar na clusterização!"
assert set(IDENTIFICADORES_EXCLUIDOS).isdisjoint(FEATURES_CLUSTER), "Identificadores não podem entrar!"

print("\nResumo da seleção final:")
identificadores_presentes = [c for c in IDENTIFICADORES_EXCLUIDOS if c in df_bruto.columns]
print(f"  - Identificadores/metadados presentes e excluídos: {len(identificadores_presentes)} -> {identificadores_presentes}")
print(f"  - Excluída (alvo - chuva):                 1 -> [{VARIAVEL_ALVO_EXCLUIDA}]")
print(f"  - Excluídas (vento - excesso de nulos):    {len(VARIAVEIS_VENTO_EXCLUIDAS)}")
print(f"  - Excluídas (extremos máx/mín redundantes):{len(VARIAVEIS_EXTREMOS_EXCLUIDAS)}")
print(f"  - Incluídas no K-Means (FEATURES_CLUSTER): {len(FEATURES_CLUSTER)} variáveis")
for f in FEATURES_CLUSTER:
    print(f"      * {f}")


---
## 2. Justificativa das Escolhas

Cada decisão de inclusão ou exclusão foi tomada avaliando **três critérios objetivos**:
1. **Relevância física e papel no problema** (evitar vazamento do alvo e ruído semântico);
2. **Redundância / multicolinearidade** (evitar dupla ou tripla contagem de uma mesma dimensão na distância euclidiana);
3. **Qualidade e dados ausentes** (evitar perda massiva de registros ou imputações artificiais).

A célula abaixo calcula o diagnóstico quantitativo (percentual de nulos e correlação de Pearson entre todas as variáveis numéricas) para embasar a tabela de justificativas.


In [ ]:
def converter_serie_numerica(serie: pd.Series) -> pd.Series:
    texto = serie.astype("string").str.strip().replace({
        "": pd.NA, "-": pd.NA, "--": pd.NA, "NaN": pd.NA, "nan": pd.NA,
        "null": pd.NA, "None": pd.NA, "-9999": pd.NA, "-9999,0": pd.NA, "-9999.0": pd.NA,
    })
    tem_virgula = texto.str.contains(",", na=False)
    texto = texto.mask(
        tem_virgula,
        texto.str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    )
    return pd.to_numeric(texto, errors="coerce").astype(float)


df_diag = df_bruto.copy()
colunas_num_originais = [c for c in df_diag.columns if c not in IDENTIFICADORES_EXCLUIDOS]
for col in colunas_num_originais:
    df_diag[col] = converter_serie_numerica(df_diag[col])

# Diagnóstico de dados ausentes em todas as colunas do CSV
diag_nulos = pd.DataFrame({
    "coluna": df_bruto.columns,
    "nulos": [
        int(df_diag[c].isna().sum()) if c in colunas_num_originais else int(df_bruto[c].isna().sum())
        for c in df_bruto.columns
    ],
})
diag_nulos["pct_nulos"] = (diag_nulos["nulos"] / len(df_bruto) * 100).round(2)
diag_nulos["decisao"] = np.where(
    diag_nulos["coluna"].isin(VARIAVEIS_FISICAS_SELECIONADAS),
    "INCLUÍDA",
    np.where(
        diag_nulos["coluna"].isin(["datetime"]),
        "USADA PARA DERIVAR CICLOS (excluída como bruta)",
        "EXCLUÍDA",
    ),
)

print("Diagnóstico de dados ausentes por coluna original:")
display(diag_nulos)

# Correlação entre medições horárias principais e seus extremos máx/mín na hora anterior
nomes_curtos = {
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)": "Chuva (mm)",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)": "Pressão",
    "PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)": "Pressão Máx",
    "PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)": "Pressão Mín",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)": "Temp. Ar",
    "TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)": "Temp. Máx",
    "TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)": "Temp. Mín",
    "TEMPERATURA DO PONTO DE ORVALHO (°C)": "P. Orvalho",
    "TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)": "Orvalho Máx",
    "TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)": "Orvalho Mín",
    "UMIDADE RELATIVA DO AR, HORARIA (%)": "Umidade",
    "UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)": "Umidade Máx",
    "UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)": "Umidade Mín",
}

cols_corr = list(nomes_curtos.keys())
corr_mat = df_diag[cols_corr].corr().rename(index=nomes_curtos, columns=nomes_curtos)

plt.figure(figsize=(11, 8))
sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, linewidths=0.4)
plt.title("Correlação de Pearson: Variáveis Horárias vs. Extremos da Hora Anterior (Redundância)")
plt.tight_layout()
plt.show()

pares_redundantes = [
    ("Pressão", "Pressão Máx"),
    ("Pressão", "Pressão Mín"),
    ("Temp. Ar", "Temp. Máx"),
    ("Temp. Ar", "Temp. Mín"),
    ("P. Orvalho", "Orvalho Máx"),
    ("P. Orvalho", "Orvalho Mín"),
    ("Umidade", "Umidade Máx"),
    ("Umidade", "Umidade Mín"),
]
print("Correlação entre cada variável selecionada e seus extremos excluídos:")
for a, b in pares_redundantes:
    print(f"  - {a:10s} vs {b:12s}: r = {corr_mat.loc[a, b]:.3f}")


### Quadro de Decisões e Justificativas

| Variável / Grupo | Decisão | Justificativa |
|---|---|---|
| Pressão, temperatura do ar, ponto de orvalho e umidade | **Incluídas** | Descrevem o estado atmosférico; o diagnóstico acima mostra suas ausências e a correlação entre medições e extremos. |
| Hora e mês cíclicos | **Incluídos** | Representam ciclos diários e anuais sem tratar seus limites como pontos distantes. |
| Precipitação horária | **Excluída do treino** | É o fenômeno usado na validação externa; incluí-la faria os clusters refletirem diretamente a chuva. |
| Timestamp e estação | **Excluídos das entradas** | `datetime` ordena a série e gera as features cíclicas; `ESTACAO` é metadado. |
| Variáveis de vento | **Excluídas** | O diagnóstico mostra cerca de 35% de ausências; incluí-las exigiria imputação substancial. |
| Extremos horários máximos e mínimos | **Excluídos** | O diagnóstico de correlação mostra alta redundância com as medições horárias principais. |


---
## 3. Tratamento de Dados e Decisão de Encoding

O CSV consolidado contém timestamps UTC, medições numéricas e metadados da estação. Ordenamos por `datetime`, removemos timestamps inválidos e duplicados, convertemos as medições para valores numéricos e preenchemos valores ausentes apenas com observações anteriores. Estados iniciais ainda sem leitura são descartados para evitar imputação com dados futuros. A precipitação ausente recebe zero somente na série auxiliar usada para validação externa.

As quatro medições físicas são contínuas e não precisam de One-Hot ou Label Encoding. Derivamos hora local de Recife e mês de `datetime` e os codificamos com seno e cosseno, preservando a proximidade entre 23h e 00h e entre dezembro e janeiro.


In [ ]:
def preparar_dados_recife(caminho_csv: str = CAMINHO_CSV) -> pd.DataFrame:
    """Carrega, converte tipos, preenche ausentes apenas com observações anteriores e gera encoding cíclico."""
    df = pd.read_csv(caminho_csv, dtype=str)

    # 1. Timestamp e ordenação cronológica
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce", utc=True)
    df = df.dropna(subset=["datetime"]).sort_values("datetime").drop_duplicates("datetime").reset_index(drop=True)

    # 2. Conversão de tipos das variáveis físicas + coluna auxiliar de chuva (apenas validação)
    for col in VARIAVEIS_FISICAS_SELECIONADAS + [VARIAVEL_ALVO_EXCLUIDA]:
        df[col] = converter_serie_numerica(df[col])

    # 3. Preenchimento causal; descarta estados iniciais ainda ausentes
    for col in VARIAVEIS_FISICAS_SELECIONADAS:
        df[col] = df[col].ffill()
    df = df.dropna(subset=VARIAVEIS_FISICAS_SELECIONADAS)

    # Chuva mantida apenas como coluna auxiliar para validação externa posterior
    df["chuva_mm"] = df[VARIAVEL_ALVO_EXCLUIDA].fillna(0.0)

    # 4. Variáveis temporais (Recife = UTC-3) e codificação cíclica
    df["datetime_local"] = df["datetime"].dt.tz_convert("America/Recife")
    df["hora_local"] = df["datetime_local"].dt.hour
    df["mes"] = df["datetime_local"].dt.month

    df["hora_sin"] = np.sin(2 * np.pi * df["hora_local"] / 24.0)
    df["hora_cos"] = np.cos(2 * np.pi * df["hora_local"] / 24.0)
    df["mes_sin"] = np.sin(2 * np.pi * df["mes"] / 12.0)
    df["mes_cos"] = np.cos(2 * np.pi * df["mes"] / 12.0)

    return df


df_tratado = preparar_dados_recife(CAMINHO_CSV)

print("Tipos das features finais (todas numéricas float64):")
display(pd.DataFrame({
    "dtype": df_tratado[FEATURES_CLUSTER].dtypes.astype(str),
    "nulos_apos_tratamento": df_tratado[FEATURES_CLUSTER].isna().sum(),
}))

print("\nEstatísticas descritivas das 8 features de entrada:")
display(df_tratado[FEATURES_CLUSTER].describe().T.round(3))

# Visualização da geometria da codificação cíclica
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.3))

horas_unicas = df_tratado[["hora_local", "hora_sin", "hora_cos"]].drop_duplicates().sort_values("hora_local")
axes[0].scatter(horas_unicas["hora_cos"], horas_unicas["hora_sin"], s=55, color="#1f77b4")
for _, r in horas_unicas.iloc[::2].iterrows():
    axes[0].annotate(f"{int(r['hora_local']):02d}h", (r["hora_cos"] * 1.13, r["hora_sin"] * 1.13), ha="center", va="center", fontsize=8)
axes[0].set_title("Codificação Cíclica da Hora Local (UTC-3)")
axes[0].set_xlabel("hora_cos")
axes[0].set_ylabel("hora_sin")
axes[0].set_xlim(-1.35, 1.35)
axes[0].set_ylim(-1.35, 1.35)
axes[0].set_aspect("equal")

meses_unicos = df_tratado[["mes", "mes_sin", "mes_cos"]].drop_duplicates().sort_values("mes")
nomes_mes = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
axes[1].scatter(meses_unicos["mes_cos"], meses_unicos["mes_sin"], s=65, color="#d95f02")
for _, r in meses_unicos.iterrows():
    axes[1].annotate(nomes_mes[int(r["mes"]) - 1], (r["mes_cos"] * 1.15, r["mes_sin"] * 1.15), ha="center", va="center", fontsize=8.5)
axes[1].set_title("Codificação Cíclica do Mês")
axes[1].set_xlabel("mes_cos")
axes[1].set_ylabel("mes_sin")
axes[1].set_xlim(-1.35, 1.35)
axes[1].set_ylim(-1.35, 1.35)
axes[1].set_aspect("equal")

plt.tight_layout()
plt.show()


---
## 4. Comparação de Normalização

O K-Means agrupa observações minimizando a distância euclidiana quadrática até os centróides. Quando as variáveis têm unidades e variâncias muito diferentes (como Pressão ~1013 mB, Umidade 30–100% e senos/cossenos em $[-1, 1]$), a escala determina qual variável domina o agrupamento.

Nesta seção executamos o K-Means ($k \in \{2, \dots, 7\}$) em três cenários:
1. **Sem Escala** (unidades originais);
2. **`StandardScaler`** (padronização $z$-score: média $0$, desvio-padrão $1$);
3. **`MinMaxScaler`** (reescala linear para $[0, 1]$).

Comparamos a **variância efetiva de cada feature**, o **Método do Cotovelo (inércia)**, o **Coeficiente de Silhueta** (calculado sobre uma amostra fixa reprodutível de 4.000 observações para comparação uniforme) e a **interpretação dos grupos** formados em $k=3$.


In [ ]:
X_raw = df_tratado[FEATURES_CLUSTER].copy()

escaladores = {
    "Sem Escala": None,
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
}

matrizes_escaladas = {}
variancias = {}
for nome, scaler in escaladores.items():
    if scaler is None:
        X_mat = X_raw.to_numpy(dtype=float)
    else:
        X_mat = scaler.fit_transform(X_raw)
    matrizes_escaladas[nome] = X_mat
    variancias[nome] = pd.Series(np.var(X_mat, axis=0, ddof=1), index=FEATURES_CLUSTER)

df_var = pd.DataFrame(variancias).round(4)
print("Variância de cada variável após cada estratégia de normalização:")
display(df_var)

# Avaliação do Cotovelo (k=1..7) e Silhueta (k=2..7)
rng = np.random.default_rng(RANDOM_STATE)
amostra_idx = rng.choice(len(X_raw), size=min(4000, len(X_raw)), replace=False)

registros_escala = []
modelos_k3 = {}
for nome, X_mat in matrizes_escaladas.items():
    for k_val in range(1, 8):
        km = KMeans(n_clusters=k_val, random_state=RANDOM_STATE, n_init=10)
        labels = km.fit_predict(X_mat)
        sil = np.nan if k_val == 1 else float(silhouette_score(X_mat[amostra_idx], labels[amostra_idx]))
        registros_escala.append({
            "normalizacao": nome,
            "k": k_val,
            "inercia": float(km.inertia_),
            "silhouette": sil,
        })
        if k_val == 3:
            modelos_k3[nome] = labels

df_comp_escala = pd.DataFrame(registros_escala)

# Tabela resumo da silhueta por k e normalização
tabela_sil = df_comp_escala.dropna(subset=["silhouette"]).pivot(index="k", columns="normalizacao", values="silhouette")[
    ["Sem Escala", "StandardScaler", "MinMaxScaler"]
].round(4)
print("\nCoeficiente de Silhueta médio (amostra fixa de 4.000 horas) por k e normalização:")
display(tabela_sil)

# Gráficos de Cotovelo (subplots individuais pela diferença de magnitude da inércia) e Silhueta
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
cores_esc = {"Sem Escala": "#7f7f7f", "StandardScaler": "#1f77b4", "MinMaxScaler": "#d95f02"}

for ax, nome in zip(axes, ["Sem Escala", "StandardScaler", "MinMaxScaler"]):
    sub = df_comp_escala[df_comp_escala["normalizacao"] == nome]
    ax.plot(sub["k"], sub["inercia"], marker="o", color=cores_esc[nome], linewidth=2)
    ax.set_title(f"Cotovelo — {nome}")
    ax.set_xlabel("Número de Clusters (k)")
    ax.set_ylabel("Inércia (SSE)")
    ax.set_xticks(range(1, 8))
plt.tight_layout()
plt.show()

plt.figure(figsize=(8.5, 4.3))
for nome in ["Sem Escala", "StandardScaler", "MinMaxScaler"]:
    sub = df_comp_escala[(df_comp_escala["normalizacao"] == nome) & (df_comp_escala["k"] >= 2)]
    plt.plot(sub["k"], sub["silhouette"], marker="o", linewidth=2, color=cores_esc[nome], label=nome)
plt.title("Comparação do Coeficiente de Silhueta por Estratégia de Normalização")
plt.xlabel("Número de Clusters (k)")
plt.ylabel("Silhueta média (no respectivo espaço)")
plt.xticks(range(2, 8))
plt.legend()
plt.tight_layout()
plt.show()

# Interpretação dos grupos para k=3 em cada estratégia de escala
print("Interpretação comparativa dos clusters (k = 3) nas unidades físicas originais:")
for nome in ["Sem Escala", "MinMaxScaler", "StandardScaler"]:
    tmp = df_tratado.copy()
    tmp["cluster"] = modelos_k3[nome]
    # Ordenar por temperatura média decrescente apenas para facilitar leitura lado a lado
    ordem = tmp.groupby("cluster")[VARIAVEIS_FISICAS_SELECIONADAS[1]].mean().sort_values(ascending=False).index
    tmp["cluster"] = tmp["cluster"].map({ant: nov for nov, ant in enumerate(ordem)})
    resumo = tmp.groupby("cluster").agg(
        horas=("chuva_mm", "size"),
        pressao_mb=("PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)", "mean"),
        temp_c=("TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)", "mean"),
        orvalho_c=("TEMPERATURA DO PONTO DE ORVALHO (°C)", "mean"),
        umid_pct=("UMIDADE RELATIVA DO AR, HORARIA (%)", "mean"),
        hora_cos_medio=("hora_cos", "mean"),
        mes_cos_medio=("mes_cos", "mean"),
        pct_com_chuva=("chuva_mm", lambda s: (s > 0).mean() * 100),
    ).round(2)
    print(f"\n--- {nome} (k = 3) ---")
    display(resumo)


### Conclusão sobre a Normalização Escolhida

- **Sem escala:** a variância da umidade (164,35) é muito maior que a de pressão (4,83), ponto de orvalho (2,04) e componentes cíclicas (cerca de 0,5). O K-Means tende a separar sobretudo pela umidade; para k=3, a silhueta amostral é 0,5107.
- **MinMaxScaler:** as componentes cíclicas têm variância próxima de 0,125, cerca de 5 a 9 vezes a variância reescalada da pressão e do ponto de orvalho. Os extremos definem a faixa e podem comprimir o miolo das medições.
- **Escolha: `StandardScaler`:** padroniza as oito features para variância unitária e dá escala comparável às dimensões físicas e temporais. Para k=3, a silhueta amostral é 0,2143; a escolha prioriza equilíbrio e interpretação, não a maior silhueta isolada.

> **Decisão registrada**: utilizar **`StandardScaler`** na pipeline de clusterização.


---
## 5. Decisão sobre PCA (Análise de Componentes Principais)

Nesta etapa investigamos se aplicar **PCA** após o `StandardScaler` melhora a qualidade dos clusters do K-Means ou se é preferível manter as 8 variáveis padronizadas originais.

Avaliamos:
1. A **variância explicada** individual e acumulada das 8 componentes principais;
2. A comparação para $k = 3$ (e $k = 4$) entre o K-Means **Sem PCA (8 dimensões)** e **Com PCA** ($2, 3, 4, 5$ componentes), verificando:
   - Silhueta medida *dentro do subespaço PCA* (que tende a inflar artificialmente por perda de variância);
   - Silhueta e índice **Davies-Bouldin** medidos de forma justa no **espaço original padronizado** (para verificar se a partição obtida via PCA melhora ou piora a coesão real nos dados);
   - Concordância das atribuições via **Adjusted Rand Index (ARI)** e separação externa de chuva.


In [ ]:
X_std = StandardScaler().fit_transform(X_raw)

pca_completo = PCA(random_state=RANDOM_STATE).fit(X_std)
var_ind = pca_completo.explained_variance_ratio_
var_acum = np.cumsum(var_ind)

df_pca_var = pd.DataFrame({
    "componente": [f"PC{i}" for i in range(1, len(FEATURES_CLUSTER) + 1)],
    "var_explicada_pct": (var_ind * 100).round(2),
    "var_acumulada_pct": (var_acum * 100).round(2),
})
print("Variância explicada por componente principal:")
display(df_pca_var)

# Gráfico Scree Plot + Variância Acumulada
fig, ax = plt.subplots(figsize=(8.5, 4.2))
x_comp = np.arange(1, len(FEATURES_CLUSTER) + 1)
ax.bar(x_comp, var_ind * 100, alpha=0.75, color="#1f77b4", label="Variância individual (%)")
ax.plot(x_comp, var_acum * 100, marker="o", color="#d95f02", linewidth=2, label="Variância acumulada (%)")
ax.axhline(80, color="gray", linestyle="--", linewidth=1, label="Referência 80%")
ax.set_title("PCA sobre as 8 Features Padronizadas (StandardScaler)")
ax.set_xlabel("Componente Principal")
ax.set_ylabel("Variância Explicada (%)")
ax.set_xticks(x_comp)
ax.set_xticklabels([f"PC{i}" for i in x_comp])
ax.legend()
plt.tight_layout()
plt.show()

# Comparação sistemática Sem PCA vs Com PCA para k=3 e k=4
comparacoes_pca = []
labels_k3_base = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10).fit_predict(X_std)
labels_k3_pca2 = None

for k_eval in [3, 4]:
    km_base = KMeans(n_clusters=k_eval, random_state=RANDOM_STATE, n_init=10).fit(X_std)
    rot_base = km_base.labels_
    comparacoes_pca.append({
        "k": k_eval,
        "configuracao": "Sem PCA (8 features)",
        "n_componentes": len(FEATURES_CLUSTER),
        "var_retida_pct": 100.0,
        "sil_espaco_treino": round(float(silhouette_score(X_std[amostra_idx], rot_base[amostra_idx])), 4),
        "sil_espaco_original": round(float(silhouette_score(X_std[amostra_idx], rot_base[amostra_idx])), 4),
        "davies_bouldin_orig": round(float(davies_bouldin_score(X_std, rot_base)), 4),
        "ari_vs_sem_pca": 1.0000,
    })

    for n_comp in [2, 3, 4, 5]:
        pca_n = PCA(n_components=n_comp, random_state=RANDOM_STATE)
        X_pca = pca_n.fit_transform(X_std)
        rot_pca = KMeans(n_clusters=k_eval, random_state=RANDOM_STATE, n_init=10).fit_predict(X_pca)
        if k_eval == 3 and n_comp == 2:
            labels_k3_pca2 = rot_pca
        comparacoes_pca.append({
            "k": k_eval,
            "configuracao": f"Com PCA ({n_comp} comps)",
            "n_componentes": n_comp,
            "var_retida_pct": round(float(pca_n.explained_variance_ratio_.sum() * 100), 1),
            "sil_espaco_treino": round(float(silhouette_score(X_pca[amostra_idx], rot_pca[amostra_idx])), 4),
            "sil_espaco_original": round(float(silhouette_score(X_std[amostra_idx], rot_pca[amostra_idx])), 4),
            "davies_bouldin_orig": round(float(davies_bouldin_score(X_std, rot_pca)), 4),
            "ari_vs_sem_pca": round(float(adjusted_rand_score(rot_base, rot_pca)), 4),
        })

df_comp_pca = pd.DataFrame(comparacoes_pca)
print("Comparação Sem PCA vs. Com PCA (avaliados no mesmo espaço original padronizado):")
display(df_comp_pca)

# Visualização comparativa em projeção 2D (K-Means sem PCA vs K-Means após PCA-2)
X_pca_vis = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_std)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, rot, titulo in [
    (axes[0], labels_k3_base, "K-Means Sem PCA (8D) — projetado em 2D"),
    (axes[1], labels_k3_pca2, "K-Means Com PCA (2D)"),
]:
    sns.scatterplot(
        x=X_pca_vis[amostra_idx, 0],
        y=X_pca_vis[amostra_idx, 1],
        hue=rot[amostra_idx],
        palette="Set1",
        s=12,
        alpha=0.55,
        ax=ax,
    )
    ax.set_title(titulo)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()


### Decisão Registrada sobre PCA

**Decisão: não usar PCA como etapa de pré-processamento**; manter PCA de duas componentes apenas para visualização.

Na base atual, as duas primeiras componentes retêm 58,8% da variância. Para k=3, a silhueta no espaço original padronizado é 0,2143 sem PCA e 0,2112 com PCA(2); para k=4, é 0,2047 e 0,1995, respectivamente. PCA(3–5) produz partições parecidas com a original (ARI de 0,86 a 0,98), sem ganho consistente de coesão. Manter as oito features facilita interpretar os grupos nas unidades meteorológicas.


---
## 6. Pipeline exploratória anterior (K=3, sem PCA)

Esta execução é histórica e exploratória: usa dados padronizados sem PCA e escolheu K=3. Ela não representa a seleção final. A seção 9 recalcula as métricas após PCA, avalia K=2 a 20 e substitui esta escolha.
1. **Seleção e tratamento (`preparar_dados_recife`)**: leitura do CSV, exclusão de `PRECIPITAÇÃO TOTAL, HORÁRIO (mm)` e identificadores, conversão numérica, preenchimento causal das séries físicas e codificação cíclica de hora local e mês.
2. **Escolha de $k$** sobre a matriz padronizada (`StandardScaler`): avaliamos em conjunto **Método do Cotovelo**, **redução marginal da inércia**, **Coeficiente de Silhueta**, **Davies-Bouldin** e **Calinski-Harabasz** para $k \in [2, 8]$, comparando em particular $k=3$ e $k=4$.
3. **Treino da pipeline exploratória (`StandardScaler` + `KMeans`)** com ordenação determinística dos rótulos por temperatura média decrescente (apenas sobre variáveis de entrada, sem usar chuva).
4. **Interpretação física, temporal e validação externa com chuva**.


In [ ]:
# =====================================================================
# ETAPA 1 DA PIPELINE: Seleção, Tratamento e Encoding
# =====================================================================
df_pipeline = preparar_dados_recife(CAMINHO_CSV)
X_pipeline = df_pipeline[FEATURES_CLUSTER].copy()
amostra_pipeline_idx = np.random.default_rng(RANDOM_STATE).choice(
    len(X_pipeline), size=min(4000, len(X_pipeline)), replace=False
)

# =====================================================================
# ETAPA 2 EXPLORATÓRIA: Escolha de k (StandardScaler, sem PCA)
# =====================================================================
X_padrao = StandardScaler().fit_transform(X_pipeline)

metricas_k = []
inercia_anterior = None
for k_val in range(1, 9):
    km = KMeans(n_clusters=k_val, random_state=RANDOM_STATE, n_init=10).fit(X_padrao)
    inercia = float(km.inertia_)
    reducao_pct = ((inercia_anterior - inercia) / inercia_anterior * 100) if inercia_anterior else np.nan
    inercia_anterior = inercia
    if k_val == 1:
        metricas_k.append({
            "k": k_val, "inercia": inercia, "reducao_inercia_pct": np.nan,
            "silhouette": np.nan, "davies_bouldin": np.nan, "calinski_harabasz": np.nan,
        })
    else:
        labels = km.labels_
        metricas_k.append({
            "k": k_val,
            "inercia": inercia,
            "reducao_inercia_pct": round(reducao_pct, 1),
            "silhouette": round(float(silhouette_score(X_padrao[amostra_pipeline_idx], labels[amostra_pipeline_idx])), 4),
            "davies_bouldin": round(float(davies_bouldin_score(X_padrao, labels)), 4),
            "calinski_harabasz": round(float(calinski_harabasz_score(X_padrao, labels)), 1),
        })

df_metricas_k = pd.DataFrame(metricas_k)
print("Métricas para seleção do número de clusters (k):")
display(df_metricas_k.round(4))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].plot(df_metricas_k["k"], df_metricas_k["inercia"], marker="o", color="#1f77b4", linewidth=2)
axes[0].axvline(3, color="#d95f02", linestyle="--", alpha=0.8, label="k = 3 exploratório")
axes[0].set_title("Método do Cotovelo (Inércia)")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inércia")
axes[0].legend()

sub_k2 = df_metricas_k[df_metricas_k["k"] >= 2]
axes[1].plot(sub_k2["k"], sub_k2["silhouette"], marker="o", color="#2ca02c", linewidth=2)
axes[1].axvline(3, color="#d95f02", linestyle="--", alpha=0.8, label="k = 3 exploratório")
axes[1].set_title("Silhueta Média por k")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhueta")
axes[1].legend()

axes[2].plot(sub_k2["k"], sub_k2["davies_bouldin"], marker="o", color="#9467bd", linewidth=2)
axes[2].axvline(3, color="#d95f02", linestyle="--", alpha=0.8, label="k = 3 exploratório")
axes[2].set_title("Índice Davies-Bouldin (menor é melhor)")
axes[2].set_xlabel("k")
axes[2].set_ylabel("Davies-Bouldin")
axes[2].legend()

plt.tight_layout()
plt.show()


### Justificativa histórica da escolha exploratória de $k = 3$
- **$k = 2$** possui silhueta ligeiramente maior (`0,2338`), mas agrupa de forma excessivamente grosseira (apenas dia/seco vs. noite/úmido), com o pior índice Davies-Bouldin (`1,6785`) e sem distinguir o regime quente-abafado de verão/outono da estação fria de inverno.
- **$k = 3$** marca a principal dobra do Cotovelo: até $k=3$ a inércia cai acentuadamente (**$-25,1\%$** em $k=2$ e **$-18,0\%$** em $k=3$), estabilizando em um platô de ganhos menores a partir de $k=4$ (**$-11,3\%$**). Além disso, entre $k \ge 3$, **$k = 3$** obtém a **maior silhueta (`0,2261`)**, o **maior índice Calinski-Harabasz (`5508,1`)** e uma queda expressiva no **Davies-Bouldin (`1,5494`)** (que volta a piorar em $k=4$ para `1,5568`), mantendo 3 regimes climáticos bem balanceados e fisicamente distintos.

Esta justificativa pertence à execução sem PCA e foi substituída pela comparação com PCA da seção 9.


In [ ]:
# =====================================================================
# ETAPA 3 DA PIPELINE: Treino Exploratório com sklearn.pipeline.Pipeline
# =====================================================================
K_EXPLORATORIO = 3

pipeline_kmeans = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(n_clusters=K_EXPLORATORIO, random_state=RANDOM_STATE, n_init=10)),
])

rotulos_brutos = pipeline_kmeans.fit_predict(X_pipeline)

# Ordenação determinística dos clusters por temperatura média decrescente (sem usar chuva)
df_pipeline["cluster_raw"] = rotulos_brutos
ordem_temp = (
    df_pipeline.groupby("cluster_raw")["TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)"]
    .mean()
    .sort_values(ascending=False)
    .index
)
mapa_clusters = {int(antigo): int(novo) for novo, antigo in enumerate(ordem_temp)}
df_pipeline["cluster"] = df_pipeline["cluster_raw"].map(mapa_clusters)

nomes_regimes = {
    0: "Cluster 0: Tarde Quente e Seca (pico Out–Jan)",
    1: "Cluster 1: Madrugada/Noite Úmida (Dez–Mai)",
    2: "Cluster 2: Regime Frio e Úmido (Jun–Out)",
}
paleta_clusters = {0: "#d95f02", 1: "#1f77b4", 2: "#1b9e77"}

# =====================================================================
# ETAPA 4 DA PIPELINE: Interpretação Física, Temporal e Validação com Chuva
# =====================================================================
perfil_fisico = df_pipeline.groupby("cluster")[VARIAVEIS_FISICAS_SELECIONADAS].mean().round(2)
perfil_fisico.columns = ["Pressão (mB)", "Temp. Ar (°C)", "P. Orvalho (°C)", "Umidade (%)"]
perfil_fisico["Horas"] = df_pipeline["cluster"].value_counts().sort_index()
perfil_fisico["Participação (%)"] = (perfil_fisico["Horas"] / len(df_pipeline) * 100).round(1)
perfil_fisico["Regime"] = [nomes_regimes[c] for c in perfil_fisico.index]

print("Perfil médio de cada cluster nas unidades originais:")
display(perfil_fisico)

# Heatmap de desvios padronizados (z-score) das médias de cada cluster em relação à média geral
cols_perfil_z = VARIAVEIS_FISICAS_SELECIONADAS
medias_cluster_orig = df_pipeline.groupby("cluster")[cols_perfil_z].mean().astype(float)
z_perfil = ((medias_cluster_orig - df_pipeline[cols_perfil_z].mean()) / df_pipeline[cols_perfil_z].std()).astype(float)
z_perfil.columns = ["Pressão", "Temp. Ar", "P. Orvalho", "Umidade"]
z_perfil.index = [f"Cluster {c}" for c in z_perfil.index]

plt.figure(figsize=(8.5, 3.3))
sns.heatmap(
    z_perfil,
    annot=medias_cluster_orig.round(1).to_numpy(dtype=float),
    fmt=".1f",
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
)
plt.title("Perfil Médio por Cluster (valores reais anotados; cor = desvio z em relação à média geral)")
plt.tight_layout()
plt.show()

# Distribuição temporal: Hora local e Mês do ano
fig, axes = plt.subplots(1, 2, figsize=(14, 4.3))
cores_lista = [paleta_clusters[c] for c in sorted(paleta_clusters)]

pd.crosstab(df_pipeline["hora_local"], df_pipeline["cluster"], normalize="index").mul(100).plot(
    kind="bar", stacked=True, color=cores_lista, width=0.88, ax=axes[0], legend=False
)
axes[0].set_title("% de Cada Cluster por Hora Local (Recife, UTC-3)")
axes[0].set_xlabel("Hora Local")
axes[0].set_ylabel("Participação (%)")

pd.crosstab(df_pipeline["mes"], df_pipeline["cluster"], normalize="index").mul(100).plot(
    kind="bar", stacked=True, color=cores_lista, width=0.85, ax=axes[1], legend=False
)
axes[1].set_title("% de Cada Cluster por Mês do Ano")
axes[1].set_xlabel("Mês")
axes[1].set_ylabel("Participação (%)")
axes[1].set_xticklabels(["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"], rotation=0)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, [nomes_regimes[c] for c in range(K_EXPLORATORIO)], loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.06), frameon=True, fontsize=8.5)
plt.tight_layout()
plt.subplots_adjust(bottom=0.20)
plt.show()

# Dispersão Temperatura vs Umidade com Centróides + Projeção PCA 2D
scaler_fit = pipeline_kmeans.named_steps["scaler"]
kmeans_fit = pipeline_kmeans.named_steps["kmeans"]
centroides_orig = scaler_fit.inverse_transform(kmeans_fit.cluster_centers_)
idx_temp = FEATURES_CLUSTER.index("TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)")
idx_umid = FEATURES_CLUSTER.index("UMIDADE RELATIVA DO AR, HORARIA (%)")

X_exploratorio_std = scaler_fit.transform(X_pipeline)
X_pca_2d = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_exploratorio_std)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for c in range(K_EXPLORATORIO):
    sub = df_pipeline[df_pipeline["cluster"] == c]
    axes[0].scatter(
        sub["TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)"],
        sub["UMIDADE RELATIVA DO AR, HORARIA (%)"],
        c=paleta_clusters[c],
        label=nomes_regimes[c],
        alpha=0.25,
        s=14,
    )
axes[0].scatter(
    centroides_orig[:, idx_temp],
    centroides_orig[:, idx_umid],
    c="black",
    marker="X",
    s=180,
    edgecolors="white",
    linewidths=1.2,
    label="Centróides",
    zorder=5,
)
for raw_id, novo_id in mapa_clusters.items():
    axes[0].annotate(
        f"C{novo_id}",
        (centroides_orig[raw_id, idx_temp] + 0.35, centroides_orig[raw_id, idx_umid] + 1.5),
        fontsize=9,
        fontweight="bold",
        color="black",
    )
axes[0].set_title("Dispersão: Temperatura vs. Umidade (com Centróides)")
axes[0].set_xlabel("Temperatura do Ar (°C)")
axes[0].set_ylabel("Umidade Relativa (%)")
axes[0].legend(fontsize=8, loc="lower left")

for c in range(K_EXPLORATORIO):
    mask = (df_pipeline["cluster"].to_numpy() == c)
    axes[1].scatter(
        X_pca_2d[mask, 0],
        X_pca_2d[mask, 1],
        c=paleta_clusters[c],
        label=f"Cluster {c}",
        alpha=0.25,
        s=14,
    )
axes[1].set_title("Projeção PCA 2D dos Clusters Exploratórios (apenas visualização)")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].legend(fontsize=8.5)

plt.tight_layout()
plt.show()

# Silhueta por Cluster na execução exploratória (amostra fixa de 4.000 pontos)
X_amostra = X_exploratorio_std[amostra_pipeline_idx]
rot_amostra = df_pipeline["cluster"].to_numpy()[amostra_pipeline_idx]
sil_amostras = silhouette_samples(X_amostra, rot_amostra)
sil_media_exploratoria = float(silhouette_score(X_amostra, rot_amostra))

fig, ax = plt.subplots(figsize=(8.5, 4.5))
y_inf = 10
for c in range(K_EXPLORATORIO):
    sil_c = np.sort(sil_amostras[rot_amostra == c])
    y_sup = y_inf + len(sil_c)
    ax.fill_betweenx(np.arange(y_inf, y_sup), 0, sil_c, facecolor=paleta_clusters[c], alpha=0.8)
    ax.text(-0.015, y_inf + len(sil_c) / 2, f"Cluster {c}", va="center", ha="right", fontweight="bold")
    y_inf = y_sup + 15
ax.axvline(sil_media_exploratoria, color="red", linestyle="--", linewidth=1.5, label=f"Silhueta média = {sil_media_exploratoria:.3f}")
ax.set_title(f"Distribuição da Silhueta por Cluster (Pipeline Exploratória, k = {K_EXPLORATORIO})")
ax.set_xlabel("Coeficiente de Silhueta")
ax.set_yticks([])
ax.set_xlim(-0.14, 0.55)
ax.legend()
plt.tight_layout()
plt.show()

# Validação externa com a variável chuva (que ficou fora do treinamento)
validacao_chuva = df_pipeline.groupby("cluster")["chuva_mm"].agg(
    horas="size",
    horas_com_chuva=lambda s: int((s > 0).sum()),
    pct_com_chuva=lambda s: round((s > 0).mean() * 100, 2),
    pct_chuva_forte_10mm=lambda s: round((s >= 10).mean() * 100, 2),
    chuva_media_mm=lambda s: round(s.mean(), 3),
    chuva_acumulada_mm=lambda s: round(s.sum(), 1),
)
validacao_chuva["Regime"] = [nomes_regimes[c] for c in validacao_chuva.index]
print("Validação externa com PRECIPITAÇÃO TOTAL (excluída do treino):")
display(validacao_chuva)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.0))
media_geral_chuva = (df_pipeline["chuva_mm"] > 0).mean() * 100
axes[0].bar(
    [f"Cluster {c}" for c in validacao_chuva.index],
    validacao_chuva["pct_com_chuva"],
    color=cores_lista,
)
axes[0].axhline(media_geral_chuva, color="red", linestyle="--", label=f"Média geral ({media_geral_chuva:.1f}%)")
axes[0].set_title("% de Horas com Chuva (> 0 mm) por Cluster")
axes[0].set_ylabel("% de Horas")
axes[0].legend()

axes[1].bar(
    [f"Cluster {c}" for c in validacao_chuva.index],
    validacao_chuva["chuva_acumulada_mm"],
    color=cores_lista,
)
axes[1].set_title("Volume Total de Chuva Acumulada (mm) por Cluster")
axes[1].set_ylabel("Precipitação Acumulada (mm)")

plt.tight_layout()
plt.show()


### Interpretação da execução exploratória (validação externa)

A pipeline exploratória sem PCA ajustou três clusters sem usar precipitação. No período completo de 2018–2021, os perfis e a validação externa são:
- **Cluster 0 — Tarde Quente e Seca (34,4%)**: 28,90 °C e 63,56% de umidade média; 2,32% das horas tiveram chuva e o acumulado foi 147,4 mm.
- **Cluster 1 — Madrugada/Noite Úmida (34,2%)**: 25,25 °C, ponto de orvalho de 22,47 °C e 84,94% de umidade média; 17,70% das horas tiveram chuva e o acumulado foi 4.180,4 mm.
- **Cluster 2 — Regime Frio e Úmido (31,4%)**: 23,49 °C, 85,50% de umidade média e pressão média de 1014,81 mB; teve a maior frequência de chuva (21,37%) e acumulou 3.178,8 mm.

Os números de precipitação são uma validação posterior; chuva não participa da matriz de clustering.


---
## 7. Verificação da pipeline exploratória e registro histórico

Esta seção verifica apenas a execução exploratória K=3 sem PCA descrita acima. A seleção final com PCA e K=5 é executada e validada na seção 9. Aqui se confirma:
- Exclusão de chuva e identificadores da matriz de treino;
- Zero valores ausentes na matriz final `X_pipeline`;
- Convergência e consistência da pipeline exploratória (`StandardScaler` + `KMeans` com `k = 3`, sem PCA no treino);
- Coerência meteorológica na validação externa com chuva.


In [ ]:
features_treino_pipeline = list(X_pipeline.columns)

assert len(df_pipeline) == len(df_tratado), f"Esperava preservar os registros válidos, obtido {len(df_pipeline)} de {len(df_tratado)}"
assert X_pipeline.isna().sum().sum() == 0, "Há valores ausentes na matriz de treino!"
assert all(pd.api.types.is_numeric_dtype(X_pipeline[c]) for c in features_treino_pipeline), "Todas as features devem ser numéricas!"
assert VARIAVEL_ALVO_EXCLUIDA not in features_treino_pipeline, "Variável chuva entrou indevidamente no treino!"
assert set(IDENTIFICADORES_EXCLUIDOS).isdisjoint(features_treino_pipeline), "Identificadores entraram no treino!"
assert len(pipeline_kmeans.steps) == 2 and pipeline_kmeans.steps[0][0] == "scaler" and pipeline_kmeans.steps[1][0] == "kmeans"
assert df_pipeline["cluster"].nunique() == K_EXPLORATORIO
assert validacao_chuva.loc[2, "pct_com_chuva"] > validacao_chuva.loc[0, "pct_com_chuva"], (
    "Regime úmido de inverno deve ter maior incidência de chuva que tarde quente e seca."
)

print("VALIDAÇÃO DA PIPELINE EXPLORATÓRIA CONCLUÍDA!")
print(f"  - Total de registros processados: {len(df_pipeline)}")
print(f"  - Features na pipeline ({len(features_treino_pipeline)}): {features_treino_pipeline}")
print(f"  - Normalização: StandardScaler")
print(f"  - PCA no treino: Não utilizado (apenas visualização 2D)")
print(f"  - Modelo exploratório: KMeans(n_clusters={K_EXPLORATORIO}, n_init=10, random_state={RANDOM_STATE})")
print(f"  - Silhueta média exploratória (k={K_EXPLORATORIO}): {sil_media_exploratoria:.4f}")
print("  - Incidência de chuva (validação externa):")
for c in validacao_chuva.index:
    row = validacao_chuva.loc[c]
    print(f"      Cluster {c}: {row['pct_com_chuva']:.2f}% horas com chuva | {row['chuva_acumulada_mm']:.1f} mm acumulados ({row['Regime']})")

# O CSV de clustering é gerado pela auditoria oficial na última seção do notebook.


---
## 8. Engenharia de Features Candidatas, Redundância (Clustering vs. Classificação), Risco de Leakage e Dataset de Trabalho

Esta seção integra a pipeline do notebook ao módulo [`ml/features.py`](../ml/features.py), consolidando todas as features candidatas derivadas até o instante $t$ e documentando:

1. **Consolidação de 29 features candidatas** (muito acima do mínimo de 8), abrangendo:
   - **Ciclos de hora e mês (4)**: `hora_sin`, `hora_cos`, `mes_sin`, `mes_cos`;
   - **Sazonalidade regional de Recife (1)**: `estacao_chuvosa` (quadra chuvosa de abril a julho);
   - **Variáveis termodinâmicas e amplitudes intra-horárias derivadas (4)**: `orvalho_depressao`, `amp_temp_1h`, `amp_pressao_1h`, `amp_umid_1h`;
   - **Tendências meteorológicas até $t$ (8)**: `pressao_delta_1h`, `pressao_delta_3h`, `pressao_delta_6h`, `temp_delta_1h`, `temp_delta_3h`, `orvalho_delta_3h`, `umid_delta_1h`, `umid_delta_3h`;
   - **Memória e acumulados de chuva observada até $t$ (12 — exclusivas da classificação)**: `chuva_1h`, `chuva_3h`, `chuva_6h`, `chuva_12h`, `chuva_24h`, `chuva_max_3h`, `chuva_max_6h`, `chuva_lag_1h`, `chuva_lag_3h`, `chuva_lag_6h`, `chuva_lag_12h`, `chuva_lag_24h`.
2. **Análise de relevância e correlação** entre candidatas, variáveis originais e alvos futuros (`precip_mm_limpa`, `acum_fut_3h`, `alvo_3h_acum_10mm`, `alvo_3h_10mm`), usando Pearson, Spearman, importância de `RandomForestClassifier` (em 2019) e ANOVA $F$ nos clusters.
3. **Avaliação de redundância ($|r| \ge 0{,}70$) e justificativa Clustering vs. Classificação**:
   - No **Clustering (K-Means)** entram apenas **8 features** (`pressao_mb`, `temp_c`, `temp_orvalho_c`, `umid_pct`, `hora_sin`, `hora_cos`, `mes_sin`, `mes_cos`), deixando de fora todas as variáveis de chuva (evita viciar os grupos pelo alvo), `orvalho_depressao` (correlação quase perfeita, $r = -0{,}9952$, com `umid_pct`), `estacao_chuvosa` (correlação $r = -0{,}8054$ com `mes_cos`) e deltas/amplitudes.
   - Na **Classificação Supervisionada** entram **33 features** (4 meteorológicas originais + 29 derivadas $\le t$), pois árvores de decisão aproveitam sinais de persistência (`chuva_*`), variação recente (`*_delta_*`) e amplitude intra-horária (`amp_*_1h`).
4. **Documentação e verificação formal de Data Leakage**:
   - **Alvos e colunas futuras proibidas em $X$**: `max_fut_2h`, `acum_fut_2h`, `max_fut_3h`, `acum_fut_3h` e todas as 8 colunas `alvo_*` (`shift(-1..-3)`).
   - **Chuva observada proibida no Clustering**: `precip_mm`, `precip_mm_limpa` e `chuva_*` são bloqueadas por `validar_sem_leakage(..., modo="clustering")`.
   - **Identificadores e metadados temporais isolados**: `Unnamed: 0`, `Data`, `Hora UTC`, `datetime`, `datetime_utc`, `datetime_local`, `ano`, `mes`, `dia`, `hora` ficam fora de $X$.
   - **Isolamento temporal de treino/validação**: seleção e importância de features usam apenas 2019, preservando 2020 como holdout cego.


In [ ]:
import sys
from pathlib import Path

ML_DIR = Path("../ml").resolve()
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import features as ml_features
import auditar_e_atualizar_dataset as ml_auditoria

# 1. Carregar a base horária consolidada sem modificar o CSV bruto
df_ml = ml_features.preparar_base("../dados/df_recife_datetime.csv", usar_open_meteo=False, filtrar_anos=False)
print(f"Base preparada via ml/features.py: {df_ml.shape[0]} linhas x {df_ml.shape[1]} colunas")
print(f"Total de features derivadas candidatas consolidadas: {len(ml_features.FEATURES_CANDIDATAS)}")

print("\nResumo estatistico das 29 features candidatas:")
display(df_ml[ml_features.FEATURES_CANDIDATAS].describe().T[["mean", "std", "min", "50%", "max"]].round(3))

# 2. Auditoria quantitativa de correlacao, relevancia e redundancia
audit = ml_features.auditar_features(df_ml)
df_rel = audit["relevancia_features"]
df_red = audit["redundancia_pares"].copy()
df_red["decisao_tecnica"] = [
    ml_auditoria._classificar_decisao_redundancia(a, b)
    for a, b in zip(df_red["variavel_a"], df_red["variavel_b"])
]

print("\nRelevancia das features (correlacoes, RF 2019, ANOVA F e entrada em Clustering vs. Classificacao):")
display(
    df_rel[[
        "feature", "tipo", "corr_precip_atual", "corr_acum_fut_3h",
        "corr_alvo_3h_acum_10mm", "rf_imp_2019_acum_10mm", "anova_f_clusters",
        "entra_clustering", "entra_classificacao"
    ]].round(4)
)

# Heatmap de correlacoes entre candidatas meteorologicas/ciclicas e alvos
vars_heatmap = [
    "pressao_mb", "temp_c", "temp_orvalho_c", "umid_pct",
    "orvalho_depressao", "amp_temp_1h", "amp_pressao_1h", "amp_umid_1h",
    "pressao_delta_3h", "temp_delta_3h", "umid_delta_3h",
    "hora_sin", "hora_cos", "mes_sin", "mes_cos", "estacao_chuvosa",
    "chuva_1h", "chuva_3h", "chuva_6h",
]
alvos_heatmap = ["precip_mm_limpa", "acum_fut_3h", "alvo_3h_acum_10mm", "alvo_3h_10mm"]
corr_sub = audit["correlacoes_pearson"].loc[vars_heatmap, ["pressao_mb", "temp_c", "temp_orvalho_c", "umid_pct", *alvos_heatmap]]

plt.figure(figsize=(10, 7.5))
sns.heatmap(corr_sub, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, linewidths=0.4)
plt.title("Correlação de Pearson: Originais & Candidatas vs. Originais & Alvos")
plt.tight_layout()
plt.show()

print(f"\nPares com alta redundancia (|r| >= 0.70): {len(df_red)} pares identificados (Top 15):")
display(df_red.head(15))

# 3. Verificacao formal anti-leakage e separacao de matrizes (identificadores, X e y)
df_proc = audit["df_com_clusters"]
df_id_clu, X_cluster_mat, y_cluster_mat = ml_features.separar_matrizes_modelo(df_proc, modo="clustering")
df_id_cls, X_classif_mat, y_classif_mat = ml_features.separar_matrizes_modelo(df_proc, alvo="3h_acum_10mm", modo="classificacao")

assert y_cluster_mat is None
assert y_classif_mat is not None
assert set(df_id_clu.columns).isdisjoint(X_cluster_mat.columns)
assert set(df_id_cls.columns).isdisjoint(X_classif_mat.columns)
assert set(ml_features.COLUNAS_ALVO).isdisjoint(X_classif_mat.columns)
assert set(ml_features.COLUNAS_AGREGADAS_FUTURAS).isdisjoint(X_classif_mat.columns)
assert set(ml_features.COLUNAS_CHUVA_OBSERVADA).isdisjoint(X_cluster_mat.columns)

# Atualizar datasets de trabalho e relatorios consolidados via pipeline oficial
ml_auditoria.main("../dados/df_recife_datetime.csv")

df_work = pd.read_csv("../dados/df_recife_features.csv")
df_clu_work = pd.read_csv("../dados/df_recife_cluster_features.csv")
df_alvos_work = pd.read_csv("../dados/df_recife_alvos.csv")

assert len(df_work) == len(df_ml) and df_work.isna().sum().sum() == 0
assert len(df_clu_work) == len(df_ml) and df_clu_work.isna().sum().sum() == 0
assert len(df_alvos_work) == len(df_ml)
assert set(ml_features.COLUNAS_ALVO).isdisjoint(df_work.columns)
assert not {"cluster", "cluster_0", "cluster_1", "cluster_2"}.intersection(df_work.columns)
assert "precip_mm_limpa" not in df_work.columns and "precip_mm_limpa" not in df_clu_work.columns

print("\nCHECKLIST FINAL DA SECAO 8 VERIFICADO:")
print(f"  - Identificadores separados:      {list(df_id_cls.columns)}")
print(f"  - Matriz X_clustering:            {X_cluster_mat.shape} (sem chuva e sem alvos)")
print(f"  - Matriz X_classificacao:         {X_classif_mat.shape} (sem identificadores e sem alvos futuros)")
print(f"  - Alvo separado (3h_acum_10mm):   {int(y_classif_mat.sum())} positivos de {int(y_classif_mat.notna().sum())} validos")
print(f"  - Dataset df_recife_features.csv: {df_work.shape}")
print(f"  - Dataset df_recife_cluster_features.csv: {df_clu_work.shape}")
print(f"  - Dataset df_recife_alvos.csv:    {df_alvos_work.shape}")


## 9. Seleção final de K e interpretação (PCA conforme orientação do professor)

Esta execução complementa e substitui a seleção exploratória de K=3 das seções anteriores. As métricas de Elbow e Silhouette foram recalculadas depois de `StandardScaler` + PCA, retendo 6 componentes (97,61% da variância). Foram avaliados K=2 a 20; chuva não entrou no ajuste. Na preparação, `ler_csv` ordena a série e preenche causalmente ausências com o último valor anterior: 109 registros tinham ponto de orvalho e umidade ausentes. Depois disso, linhas ainda incompletas são removidas.

| K | Silhouette | Davies–Bouldin | Redução da inércia |
|---:|---:|---:|---:|
| 2 | 0,2382 | 1,6541 | 25,62% |
| 3 | 0,2206 | 1,5632 | 17,35% |
| 5 | 0,2242 | 1,4527 | 11,48% |
| 10 | 0,2263 | 1,3238 | 6,43% |

**K final: 5.** K=2 tem a maior Silhouette, mas resume a série em apenas dois regimes amplos. K=5 mantém Silhouette em 0,2242, melhora em relação a K=3 e reduz Davies–Bouldin para 1,4527. As cinco partições têm entre 16,68% e 22,77% dos registros. K=10 melhora a Silhouette em apenas 0,0021 sobre K=5, com o dobro de grupos; a redução marginal da inércia também cai de 11,48% em K=5 para 8,61% em K=6.

Os IDs estão ordenados por temperatura média crescente; a interpretação usa os atributos meteorológicos originais:

- **Fresco e muito úmido (21,58%)**: 22,73 °C, umidade média de 89,12% e pressão de 1014,99 mB; é o grupo mais fresco e de maior pressão.
- **Morno e abafado (22,77%)**: 25,06 °C, ponto de orvalho de 22,87 °C e umidade de 87,89%; combina temperatura amena com muito vapor d'água.
- **Morno e umidade intermediária (18,71%)**: 25,43 °C e umidade de 78,64%; faz a transição entre os grupos mais úmidos e os mais secos.
- **Quente e menos úmido (16,68%)**: 27,29 °C, umidade de 67,09% e pressão média relativamente alta (1014,44 mB).
- **Muito quente e mais seco (20,25%)**: 29,75 °C, umidade de 62,27% e ponto de orvalho de 21,66 °C; a temperatura sobe sem aumento equivalente do ponto de orvalho.

O relatório contém médias, medianas, desvio-padrão e IQR de cada atributo por grupo, atribuições por registro, métricas completas para todos os K e os gráficos exportados em PNG/PDF: [relatório](../reports/kmeans_recife/relatorio.md). Para reproduzir a execução a partir da raiz: `venv/bin/python ml/experimento_kmeans.py`.

![Elbow e métricas](../reports/kmeans_recife/figuras/criterios_k.png)

![Silhouette por cluster](../reports/kmeans_recife/figuras/silhueta_final.png)

![Perfil meteorológico dos grupos](../reports/kmeans_recife/figuras/perfil_meteorologico.png)


In [ ]:
from IPython.display import Markdown, display
import importlib
import sys
from pathlib import Path

ROOT_KMEANS = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / "ml" / "experimento_kmeans.py").is_file()), None)
if ROOT_KMEANS is None:
    raise FileNotFoundError("Não encontrei ml/experimento_kmeans.py nos diretórios ancestrais.")
if str(ROOT_KMEANS / "ml") not in sys.path:
    sys.path.insert(0, str(ROOT_KMEANS / "ml"))

import experimento_kmeans
importlib.reload(experimento_kmeans)
experimento_kmeans.main()

relatorio = (ROOT_KMEANS / "reports" / "kmeans_recife" / "relatorio.md").read_text(encoding="utf-8")
display(Markdown(relatorio.replace("](figuras/", "](../reports/kmeans_recife/figuras/")))
